# Desafio Motiva --- Escadas rolantes (Dynamic Programming)
## Notebook de execução --- Sprint 1

Este notebook roda, passo a passo, todas as entregas do desafio:

| Parte | O que é feito | Entrega |
|---|---|---|
| Etapa 0 | Geração do dataset sintético (`gerar_dataset`) | `sensores.csv` + resumo |
| Atividade 1 | Busca linear × busca binária, contando comparações | tabela de comparações |
| Atividade 2 | Percurso na linha com pilha (`tempo_ate`, `rota_ida_e_volta`) | `linha.csv`, `tempos_base.csv` |
| Atividade 3 | Plano do turno com fila e pilha (`planejar_turno`) | plano de 4, 6 e 8 h + pendentes |

Tudo em Python puro: buscas escritas à mão, fila e pilha feitas com listas (`append`, `pop(0)`, `pop()`). Não usamos grafos, `deque` nem bibliotecas prontas de busca.

**Como rodar:** abra o notebook na pasta do projeto (Jupyter) ou, no Colab, envie antes os arquivos `gerar_dataset.py`, `dados.py`, `busca.py`, `linha.py`, `turno.py`, `main.py` e `linha.csv`. Depois, rode as células em ordem.

In [1]:
import os, sys

arquivos = ["gerar_dataset.py", "dados.py", "busca.py", "linha.py", "turno.py", "main.py", "linha.csv"]
faltando = [a for a in arquivos if not os.path.exists(a)]
print("Arquivos do projeto que faltam na pasta:", faltando or "nenhum")


def mostrar_tabela(linhas, colunas, titulos=None):
    titulos = titulos or colunas
    larguras = [max(len(str(t)), *(len(str(l[c])) for l in linhas)) for c, t in zip(colunas, titulos)]
    print(" | ".join(str(t).ljust(w) for t, w in zip(titulos, larguras)))
    print("-+-".join("-" * w for w in larguras))
    for l in linhas:
        print(" | ".join(str(l[c]).ljust(w) for c, w in zip(colunas, larguras)))

Arquivos do projeto que faltam na pasta: nenhum


---
## Etapa 0 — Dataset sintético

**Regras da geração** (`gerar_dataset(n_escadas, n_leituras, semente)` em `gerar_dataset.py`):

- `random.seed(semente)` no início: com a mesma semente, o arquivo sai sempre igual. **Semente usada: 42.**
- **50 escadas** (o enunciado aceita de 30 a 197; a função recusa valores fora disso) e **200 leituras por escada**, uma por hora, a partir de 01/01/2026 00:00.
- Cada escada fica numa das 11 estações da Linha 4-Amarela e é sorteada como **crítica** ou **não crítica**.
- IDs com 3 dígitos (`ESC-001` … `ESC-050`), para a ordem do texto ser igual à ordem do número (isso importa na busca binária).
- **Perfis:** 60% saudáveis, 30% em desgaste gradual e 10% com falhas pontuais.
  - *Saudável:* todos os sensores sorteados dentro da faixa normal.
  - *Desgaste:* vibração sobe até +4 mm/s, temperatura até +20 °C e o óleo cai até −30 pontos, aos poucos, conforme as horas passam.
  - *Falha pontual:* normal na maior parte do tempo, mas cada leitura tem 5% de chance de um pico de vibração (7–10 mm/s) e temperatura (75–90 °C).
- **Ordem no arquivo:** as horas estão em ordem, mas dentro de cada hora as escadas são embaralhadas.

Faixas normais: vibração 1–4 · temperatura 35–65 · corrente 10–25 · tensão 361–399 · óleo 60–100 · velocidade 0,48–0,52.

In [2]:
from gerar_dataset import gerar_dataset, resumir_dataset, FAIXAS

SEMENTE = 42
leituras_geradas, escadas = gerar_dataset(n_escadas=50, n_leituras=200, semente=SEMENTE, caminho_saida="sensores.csv")
resumo = resumir_dataset(leituras_geradas, escadas)

linhas_resumo = []
for sensor, v in resumo["sensores"].items():
    minimo, maximo = FAIXAS[sensor]
    linhas_resumo.append({"sensor": sensor, "faixa": f"{minimo}–{maximo}",
                          "min": round(v["minimo"], 2), "media": round(v["media"], 2), "max": round(v["maximo"], 2)})
print("Resumo por sensor\n")
mostrar_tabela(linhas_resumo, ["sensor", "faixa", "min", "media", "max"],
               ["Sensor", "Faixa normal", "Mínimo", "Média", "Máximo"])

print("\nEscadas por perfil:", resumo["perfis"])
print("Escadas por grupo: ", resumo["grupos"])
print("Total de leituras: ", resumo["total_leituras"])

Resumo por sensor

Sensor      | Faixa normal | Mínimo | Média  | Máximo
------------+--------------+--------+--------+-------
vibracao    | 1–4          | 1.0    | 3.13   | 9.78  
temperatura | 35–65        | 35.0   | 53.14  | 89.2  
corrente    | 10–25        | 10.0   | 17.47  | 25.0  
tensao      | 361–399      | 361.0  | 380.11 | 399.0 
nivel_oleo  | 60–100       | 30.6   | 75.73  | 100.0 
velocidade  | 0.48–0.52    | 0.48   | 0.5    | 0.52  

Escadas por perfil: {'saudavel': 30, 'desgaste': 15, 'falha': 5}
Escadas por grupo:  {'crítica': 24, 'não crítica': 26}
Total de leituras:  10000


In [3]:
mostrar_tabela(leituras_geradas[:6], ["id_escada", "estacao", "grupo", "data_hora", "vibracao", "temperatura", "nivel_oleo"])

id_escada | estacao                | grupo       | data_hora        | vibracao | temperatura | nivel_oleo
----------+------------------------+-------------+------------------+----------+-------------+-----------
ESC-030   | Higienópolis-Mackenzie | crítica     | 2026-01-01 00:00 | 2.31     | 50.5        | 73.5      
ESC-029   | Higienópolis-Mackenzie | não crítica | 2026-01-01 00:00 | 1.69     | 41.6        | 69.2      
ESC-006   | República              | crítica     | 2026-01-01 00:00 | 3.58     | 37.1        | 68.6      
ESC-014   | Vila Sônia             | não crítica | 2026-01-01 00:00 | 2.71     | 49.2        | 67.6      
ESC-038   | Vila Sônia             | crítica     | 2026-01-01 00:00 | 2.29     | 47.7        | 86.9      
ESC-025   | República              | crítica     | 2026-01-01 00:00 | 1.3      | 47.1        | 69.9      


**Interpretação.** Corrente, tensão e velocidade ficam sempre dentro da faixa, porque nenhum perfil mexe nelas. Vibração, temperatura e óleo passam da faixa por causa dos perfis de desgaste (que pioram aos poucos) e de falha (picos isolados). A média de vibração (≈3,1) é maior que o meio da faixa (2,5) justamente por causa das escadas em desgaste. A divisão 30 / 15 / 5 bate com os 60% / 30% / 10% pedidos.

---
## Atividade 1 — Busca linear × busca binária

Uma leitura está **em alerta** quando algum sensor sai da faixa normal (o limite conta como normal). As funções `busca_linear` e `busca_binaria` (em `busca.py`) foram escritas à mão e retornam `(posição, comparações)`.

- **Linear:** olha um item por vez, do começo ao fim. 1 comparação por item.
- **Binária:** precisa de lista ordenada; olha o meio e descarta metade. Contamos 2 comparações por passo (`==` e `<`).

### Parte A — localizar escadas pelo ID (1.000 consultas)

Sorteamos 1.000 IDs com semente 42. Cada ID é procurado de três formas: linear na lista do arquivo (10.000 leituras), linear na lista de IDs e binária na lista de IDs ordenada. O código **confere** se cada busca achou a escada certa — se não achar, para com erro.

In [4]:
from dados import ler_leituras, agrupar_por_escada, esta_em_alerta, escadas_em_alerta
from busca import busca_linear, busca_binaria, ordenar_ids
from main import sortear_consultas, comparar_buscas_por_id, resumir_comparacoes, comparar_primeira_leitura

leituras = ler_leituras("sensores.csv")
por_escada = agrupar_por_escada(leituras)
ids_ordenados = ordenar_ids(por_escada.keys())
lista_ordenada = [{"id": i} for i in ids_ordenados]
print("Leituras:", len(leituras), "| escadas:", len(ids_ordenados), "| primeiros IDs:", ids_ordenados[:4])

consultas = sortear_consultas(ids_ordenados, 1000, 42)
por_consulta = comparar_buscas_por_id(leituras, lista_ordenada, consultas)
print("Todas as 1.000 consultas acharam a escada certa nas três buscas.\n")

print("Tabela de comparações (1.000 consultas)\n")
mostrar_tabela(resumir_comparacoes(por_consulta), ["metodo", "total", "media", "maximo"],
               ["Método", "Total de comparações", "Média", "Máximo"])

Leituras: 10000 | escadas: 50 | primeiros IDs: ['ESC-001', 'ESC-002', 'ESC-003', 'ESC-004']
Todas as 1.000 consultas acharam a escada certa nas três buscas.

Tabela de comparações (1.000 consultas)

Método                     | Total de comparações | Média | Máximo
---------------------------+----------------------+-------+-------
Linear na lista do arquivo | 25127                | 25.13 | 50    
Linear na lista de IDs     | 25835                | 25.84 | 50    
Binária na lista de IDs    | 8788                 | 8.79  | 11    


In [5]:
print("Primeiras 10 consultas\n")
mostrar_tabela(por_consulta[:10], ["consulta", "id_escada", "comp_linear_arquivo", "comp_linear_ids", "comp_binaria"],
               ["#", "ID", "Linear (arquivo)", "Linear (IDs)", "Binária (IDs)"])

Primeiras 10 consultas

#  | ID      | Linear (arquivo) | Linear (IDs) | Binária (IDs)
---+---------+------------------+--------------+--------------
1  | ESC-041 | 43               | 41           | 7            
2  | ESC-008 | 24               | 8            | 11           
3  | ESC-002 | 23               | 2            | 11           
4  | ESC-048 | 15               | 48           | 11           
5  | ESC-018 | 10               | 18           | 5            
6  | ESC-016 | 31               | 16           | 9            
7  | ESC-015 | 11               | 15           | 7            
8  | ESC-009 | 12               | 9            | 7            
9  | ESC-048 | 15               | 48           | 11           
10 | ESC-007 | 44               | 7            | 9            


### Parte B — primeira leitura em que os alertas acumulados chegam a 5

Para cada escada, transformamos as 200 leituras (em ordem de horário) em 0/1 (alerta ou não).

- **Linear:** vai somando os alertas até a soma chegar a 5.
- **Binária:** a soma acumulada nunca diminui (é uma lista ordenada!), então a binária acha a primeira posição em que ela vale 5 ou mais.

As duas buscas precisam achar a mesma posição; o código confere isso para cada escada.

In [6]:
primeira = comparar_primeira_leitura(por_escada, ids_ordenados, 5)
chegam = [l for l in primeira if l["posicao_primeira_leitura"] >= 0]

print(f"Escadas que chegam a 5 alertas: {len(chegam)} de {len(primeira)}\n")
mostrar_tabela(chegam, ["id_escada", "posicao_primeira_leitura", "data_hora", "comp_linear", "comp_binaria"],
               ["Escada", "Posição (0–199)", "Data e hora", "Comp. linear", "Comp. binária"])

total_lin = sum(l["comp_linear"] for l in primeira)
total_bin = sum(l["comp_binaria"] for l in primeira)
total_acum = sum(l["passos_montar_acumulado"] for l in primeira)
print(f"\nTodas as 50 escadas -> linear: {total_lin} | binária: {total_bin} "
      f"| passos para montar a soma acumulada (a binária precisa dela): {total_acum}")

Escadas que chegam a 5 alertas: 20 de 50

Escada  | Posição (0–199) | Data e hora      | Comp. linear | Comp. binária
--------+-----------------+------------------+--------------+--------------
ESC-006 | 89              | 2026-01-04 17:00 | 90           | 8            
ESC-011 | 29              | 2026-01-02 05:00 | 30           | 8            
ESC-015 | 34              | 2026-01-02 10:00 | 35           | 7            
ESC-016 | 16              | 2026-01-01 16:00 | 17           | 8            
ESC-019 | 17              | 2026-01-01 17:00 | 18           | 8            
ESC-021 | 25              | 2026-01-02 01:00 | 26           | 7            
ESC-023 | 20              | 2026-01-01 20:00 | 21           | 8            
ESC-024 | 68              | 2026-01-03 20:00 | 69           | 7            
ESC-025 | 42              | 2026-01-02 18:00 | 43           | 8            
ESC-026 | 35              | 2026-01-02 11:00 | 36           | 8            
ESC-030 | 27              | 2026-01-02 03:00 |

**Interpretação.**

- **Por ID:** a binária faz em média ≈9 comparações contra ≈26 da linear na mesma lista de 50 IDs. Isso bate com a teoria: log₂(50) ≈ 5,6 passos, com 2 comparações por passo → no máximo 11–12. A linear, no pior caso, olha os 50. Com as 197 escadas permitidas, a linear iria a ≈99 de média e a binária só subiria para ≈14.
- A linear **no arquivo** tem média parecida com a linear nos IDs. Isso acontece porque a primeira hora do arquivo já tem todas as 50 escadas (embaralhadas), então qualquer ID aparece nas 50 primeiras linhas. É uma característica do arquivo, não do algoritmo.
- **Primeira leitura com 5 alertas:** a binária faz ≈8 comparações por escada, contra até 200 da linear. Mas atenção: ela só funciona porque antes montamos a soma acumulada, que custa 1 passo por leitura (200 por escada). Se a busca for feita **uma vez só**, a linear sai mais barata, porque ela para assim que chega a 5. A binária compensa quando a mesma soma acumulada é consultada várias vezes (por exemplo, "quando chegou a 5? e a 10? e a 20?").
- **Cuidado que descobrimos:** a binária só funciona se a lista estiver ordenada pelo mesmo critério da comparação. Com IDs sem zeros (`ESC-2`, `ESC-10`), a ordem do número é diferente da ordem do texto, e a binária deixava de achar 10 das 50 escadas. Por isso os IDs têm 3 dígitos e o código confere cada resultado.

---
## Atividade 2 — Percurso na linha com pilha

`linha.csv` tem as **11 estações da Linha 4-Amarela** (ViaQuatro/Motiva), na ordem da linha, e os minutos até a estação seguinte (premissa do grupo). A base, **Luz**, fica na primeira posição.

- `tempo_ate(linha, estacao)`: busca linear pela estação, somando os minutos desde a base.
- `rota_ida_e_volta(linha, estacao)`: na ida, empilha as estações com `append`; na volta, desempilha com `pop()`. Os minutos da volta são somados durante o `pop()`, trecho por trecho.

In [7]:
from linha import ler_linha, tempo_ate, rota_ida_e_volta
from main import tempos_da_base

linha = ler_linha("linha.csv")
base = linha[0]["estacao"]
print("Base:", base, "| estações:", len(linha), "\n")
mostrar_tabela(linha, ["estacao", "minutos_ate_proxima"], ["Estação", "Minutos até a seguinte"])

Base: Luz | estações: 11 

Estação                | Minutos até a seguinte
-----------------------+-----------------------
Luz                    | 2                     
República              | 2                     
Higienópolis-Mackenzie | 2                     
Paulista               | 2                     
Oscar Freire           | 2                     
Fradique Coutinho      | 2                     
Faria Lima             | 2                     
Pinheiros              | 3                     
Butantã                | 3                     
São Paulo-Morumbi      | 3                     
Vila Sônia             | 0                     


In [8]:
tempos = tempos_da_base(linha)
print("tempos_base.csv\n")
mostrar_tabela(tempos, ["estacao", "minutos_ida", "minutos_ida_volta"], ["Estação", "Minutos de ida", "Minutos de ida e volta"])

tempos_base.csv

Estação                | Minutos de ida | Minutos de ida e volta
-----------------------+----------------+-----------------------
Luz                    | 0              | 0                     
República              | 2              | 4                     
Higienópolis-Mackenzie | 4              | 8                     
Paulista               | 6              | 12                    
Oscar Freire           | 8              | 16                    
Fradique Coutinho      | 10             | 20                    
Faria Lima             | 12             | 24                    
Pinheiros              | 14             | 28                    
Butantã                | 17             | 34                    
São Paulo-Morumbi      | 20             | 40                    
Vila Sônia             | 23             | 46                    


In [9]:
rota = rota_ida_e_volta(linha, "Vila Sônia")
print("Ida   (append):", " -> ".join(rota["ida"]))
print("Volta (pop())  :", " -> ".join(rota["volta"]))
print(f"Ida {rota['minutos_ida']} min + volta {rota['minutos_volta']} min = {rota['minutos_ida_volta']} min")

Ida   (append): Luz -> República -> Higienópolis-Mackenzie -> Paulista -> Oscar Freire -> Fradique Coutinho -> Faria Lima -> Pinheiros -> Butantã -> São Paulo-Morumbi -> Vila Sônia
Volta (pop())  : Vila Sônia -> São Paulo-Morumbi -> Butantã -> Pinheiros -> Faria Lima -> Fradique Coutinho -> Oscar Freire -> Paulista -> Higienópolis-Mackenzie -> República -> Luz
Ida 23 min + volta 23 min = 46 min


**Interpretação.** A pilha devolve a volta exatamente na ordem inversa da ida, sem precisar inverter a lista: o último a entrar (o destino) é o primeiro a sair. Como os trechos são os mesmos nos dois sentidos, a volta leva o mesmo tempo da ida. A estação mais distante (Vila Sônia) custa 46 min só de deslocamento — quase o tempo de um serviço normal (40 min). Isso pesa muito na Atividade 3.

---
## Atividade 3 — Plano do turno com fila e pilha

`planejar_turno(base, horas)` (em `turno.py`):

1. Monta `fila_critica` e `fila_normal` (listas) com as escadas em alerta, na **ordem do primeiro alerta**.
2. Atende a fila crítica antes da normal, retirando com `pop(0)` (fila: o primeiro que entra é o primeiro que sai).
3. **Custo = ida + serviço + volta**, com serviço de 60 min (crítica) ou 40 min (não crítica).
4. Se cabe no que sobra do turno, `append` na agenda; se não cabe, vai para **pendentes** e tentamos a próxima.
5. **Perda de 1 hora** (`perder_hora`): desfaz a agenda com `pop()` (pilha: o último agendado sai primeiro) até caber.

Premissa: o turno começa às 08:00, sem pausas, e a equipe volta à base depois de cada atendimento.

In [10]:
from turno import montar_filas, planejar_turno, perder_hora

em_alerta = escadas_em_alerta(por_escada)
fila_critica, fila_normal = montar_filas(em_alerta)
print(f"Escadas em alerta: {len(em_alerta)} (crítica: {len(fila_critica)}, não crítica: {len(fila_normal)})\n")
print("Fila crítica (ordem do primeiro alerta)")
mostrar_tabela(fila_critica, ["id_escada", "estacao", "primeiro_alerta"], ["Escada", "Estação", "Primeiro alerta"])
print("\nFila normal (ordem do primeiro alerta)")
mostrar_tabela(fila_normal, ["id_escada", "estacao", "primeiro_alerta"], ["Escada", "Estação", "Primeiro alerta"])

Escadas em alerta: 20 (crítica: 10, não crítica: 10)

Fila crítica (ordem do primeiro alerta)
Escada  | Estação                | Primeiro alerta 
--------+------------------------+-----------------
ESC-050 | Pinheiros              | 2026-01-01 02:00
ESC-016 | São Paulo-Morumbi      | 2026-01-01 05:00
ESC-034 | Fradique Coutinho      | 2026-01-01 06:00
ESC-037 | República              | 2026-01-01 08:00
ESC-030 | Higienópolis-Mackenzie | 2026-01-01 09:00
ESC-025 | República              | 2026-01-01 15:00
ESC-006 | República              | 2026-01-01 17:00
ESC-015 | Vila Sônia             | 2026-01-01 18:00
ESC-040 | São Paulo-Morumbi      | 2026-01-01 18:00
ESC-036 | República              | 2026-01-01 21:00

Fila normal (ordem do primeiro alerta)
Escada  | Estação           | Primeiro alerta 
--------+-------------------+-----------------
ESC-031 | Butantã           | 2026-01-01 01:00
ESC-026 | São Paulo-Morumbi | 2026-01-01 04:00
ESC-021 | Vila Sônia        | 2026-01-01 06:00
ESC-019

In [11]:
def mostrar_plano(plano, titulo):
    print(f"===== {titulo}: {len(plano['agenda'])} na agenda, {len(plano['pendentes'])} pendentes, "
          f"{plano['tempo_usado_min']} de {plano['limite_min']} min usados =====")
    agenda = [dict(item, rota=" > ".join(item["rota_ida"])) for item in plano["agenda"]]
    if agenda:
        mostrar_tabela(agenda, ["id_escada", "grupo", "estacao", "rota", "custo_min", "inicio", "fim"],
                       ["Escada", "Grupo", "Estação", "Rota de ida (volta = inversa)", "Custo", "Início", "Fim"])
    print("Pendentes:", ", ".join(f"{p['id_escada']} ({p['grupo']}, {p['custo_min']} min)" for p in plano["pendentes"]) or "nenhum")
    print()


planos = {}
for horas in (4, 6, 8):
    planos[horas] = planejar_turno(base, horas)
    mostrar_plano(planos[horas], f"Turno de {horas} h")

===== Turno de 4 h: 3 na agenda, 17 pendentes, 228 de 240 min usados =====
Escada  | Grupo       | Estação           | Rota de ida (volta = inversa)                                                                                                                 | Custo | Início | Fim  
--------+-------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+-------+--------+------
ESC-050 | crítica     | Pinheiros         | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros                               | 88    | 08:00  | 09:28
ESC-016 | crítica     | São Paulo-Morumbi | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros > Butantã > São Paulo-Morumbi | 100   | 09:28  | 11:08
ESC-023 | não crítica | Luz               | Luz                                  

===== Turno de 6 h: 4 na agenda, 16 pendentes, 332 de 360 min usados =====
Escada  | Grupo   | Estação           | Rota de ida (volta = inversa)                                                                                                                 | Custo | Início | Fim  
--------+---------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+-------+--------+------
ESC-050 | crítica | Pinheiros         | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros                               | 88    | 08:00  | 09:28
ESC-016 | crítica | São Paulo-Morumbi | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros > Butantã > São Paulo-Morumbi | 100   | 09:28  | 11:08
ESC-034 | crítica | Fradique Coutinho | Luz > República > Higienópolis-Mackenzie > Paulista > Osc

===== Turno de 8 h: 6 na agenda, 14 pendentes, 464 de 480 min usados =====
Escada  | Grupo   | Estação                | Rota de ida (volta = inversa)                                                                                                                 | Custo | Início | Fim  
--------+---------+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+-------+--------+------
ESC-050 | crítica | Pinheiros              | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros                               | 88    | 08:00  | 09:28
ESC-016 | crítica | São Paulo-Morumbi      | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros > Butantã > São Paulo-Morumbi | 100   | 09:28  | 11:08
ESC-034 | crítica | Fradique Coutinho      | Luz > República > Higienópolis-M

In [12]:
for horas, plano in planos.items():
    mostrar_plano(perder_hora(plano), f"Turno de {horas} h perdendo 1 hora ({horas - 1} h)")

===== Turno de 4 h perdendo 1 hora (3 h): 1 na agenda, 19 pendentes, 88 de 180 min usados =====
Escada  | Grupo   | Estação   | Rota de ida (volta = inversa)                                                                                   | Custo | Início | Fim  
--------+---------+-----------+-----------------------------------------------------------------------------------------------------------------+-------+--------+------
ESC-050 | crítica | Pinheiros | Luz > República > Higienópolis-Mackenzie > Paulista > Oscar Freire > Fradique Coutinho > Faria Lima > Pinheiros | 88    | 08:00  | 09:28
Pendentes: ESC-034 (crítica, 80 min), ESC-037 (crítica, 64 min), ESC-030 (crítica, 68 min), ESC-025 (crítica, 64 min), ESC-006 (crítica, 64 min), ESC-015 (crítica, 106 min), ESC-040 (crítica, 100 min), ESC-036 (crítica, 64 min), ESC-031 (não crítica, 74 min), ESC-026 (não crítica, 80 min), ESC-021 (não crítica, 86 min), ESC-019 (não crítica, 64 min), ESC-032 (não crítica, 80 min), ESC-042 (não 

In [13]:
resumo_turnos = []
for horas, plano in planos.items():
    depois = perder_hora(plano)
    resumo_turnos.append({"turno": f"{horas} h", "agenda": len(plano["agenda"]), "pendentes": len(plano["pendentes"]),
                          "usado": f"{plano['tempo_usado_min']}/{plano['limite_min']}",
                          "agenda_menos_1h": len(depois["agenda"]), "usado_menos_1h": f"{depois['tempo_usado_min']}/{depois['limite_min']}"})
mostrar_tabela(resumo_turnos, ["turno", "agenda", "pendentes", "usado", "agenda_menos_1h", "usado_menos_1h"],
               ["Turno", "Na agenda", "Pendentes", "Min usados", "Agenda (−1 h)", "Min usados (−1 h)"])

Turno | Na agenda | Pendentes | Min usados | Agenda (−1 h) | Min usados (−1 h)
------+-----------+-----------+------------+---------------+------------------
4 h   | 3         | 17        | 228/240    | 1             | 88/180           
6 h   | 4         | 16        | 332/360    | 3             | 268/300          
8 h   | 6         | 14        | 464/480    | 5             | 400/420          


**Interpretação.**

- São 20 escadas em alerta (10 críticas e 10 não críticas), com custo de 40 a 106 min cada. Nem o turno de 8 h atende todas as críticas: ele atende 6 e deixa 14 pendentes. O gargalo é a **capacidade do turno**, não a ordem.
- No turno de 4 h, uma escada **não crítica** (ESC-023, na própria base Luz, 40 min) entra na agenda enquanto críticas ficam pendentes. Isso acontece porque, quando uma escada não cabe, a regra tenta a próxima. As críticas que sobraram custam 64 min ou mais e não cabem nos 52 min que sobram depois das duas primeiras; a ESC-023 cabe. Ou seja: a fila garante a **ordem de tentativa**, não que toda crítica seja atendida antes de qualquer normal.
- Na perda de 1 hora, o `pop()` tira primeiro o último agendado, que é o de menor prioridade. No turno de 4 h isso tira duas escadas (a ESC-023 e a ESC-016), porque tirar só a primeira ainda deixa 188 min, acima dos 180 do novo limite. Sobram 92 min livres que a regra do enunciado não tenta preencher de novo.
- Escadas longe da base (São Paulo-Morumbi, Vila Sônia) gastam até 46 min só de deslocamento. Uma segunda base no meio da linha (ex.: Paulista ou Pinheiros) reduziria bastante esse custo — é uma sugestão para a próxima sprint.

---
## Conclusão

1. **Etapa 0:** dataset com 50 escadas × 200 leituras (semente 42), perfis 30 / 15 / 5 como pedido.
2. **Atividade 1:** a binária faz ≈3× menos comparações que a linear para achar um ID, e a vantagem cresce com o tamanho da lista. Ela exige lista ordenada pelo mesmo critério da comparação.
3. **Atividade 2:** a pilha gera a rota de volta na ordem inversa naturalmente; o tempo de ida e volta vai de 0 (base) a 46 min (Vila Sônia).
4. **Atividade 3:** a fila define a prioridade (críticas primeiro, por ordem de alerta), mas a capacidade do turno limita muito: 3, 4 e 6 escadas atendidas em 4, 6 e 8 h.

**Limitações:** dados fictícios; minutos entre estações e tempos de serviço são premissas do grupo; uma equipe e uma base; regra gulosa (não busca a combinação ótima de escadas).